# Week 7 Exercise — Open-Source Base Model Price Predictor

**"The Price is Right"** — Use an **open-source** model (Llama 3.2–style) in a **zero-shot** setup with the same prompt format as Week 7 training. Compare to a simple baseline and evaluate MAE on a small in-notebook set.

*(Full QLoRA fine-tuning runs in Colab; this notebook demonstrates the prompt format and evaluation side.)*

Set `OPENROUTER_API_KEY` in `.env`. Optional: run Ollama with `llama3.2` for local inference.

In [ ]:
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")

if not api_key:
    print("Add OPENROUTER_API_KEY to .env")
else:
    print("OpenRouter API key loaded.")

# Open-source style models (same prompt format as Week 7)
MODELS = {
    "OpenRouter: Llama 3.2 3B": ("meta-llama/llama-3.2-3b-instruct", "https://openrouter.ai/api/v1", api_key),
    "OpenRouter: Qwen 2.5 3B": ("qwen/qwen-2.5-3b-instruct", "https://openrouter.ai/api/v1", api_key),
    "Ollama: llama3.2": ("llama3.2", "http://localhost:11434/v1", "ollama"),
}

def get_client(model_key):
    model_id, base_url, key = MODELS.get(model_key, list(MODELS.values())[0])
    return OpenAI(api_key=key or "ollama", base_url=base_url), model_id

## Week 7 prompt format

Same as course Day 2 / training: estimate price from product summary; respond with price only.

In [ ]:
def build_prompt(summary):
    return f"Estimate the price of this product. Respond with the price, no explanation\n\n{summary}"

def parse_price(text):
    if not text:
        return 0.0
    text = text.strip().replace(",", "")
    match = re.search(r"(\\d+(?:\\.\\d+)?)", text)
    if match:
        try:
            return max(0.0, float(match.group(1)))
        except ValueError:
            pass
    return 0.0

## Eval set

Small (summary, price) list; no dependency on week7 pricer or HuggingFace.

In [ ]:
EVAL_EXAMPLES = [
    ("Wireless Bluetooth earbuds, 24hr battery, noise cancelling", 49.99),
    ("Stainless steel water bottle 32oz insulated", 29.99),
    ("Mechanical keyboard RGB backlit, cherry MX switches", 129.00),
    ("USB-C laptop docking station dual 4K HDMI", 189.99),
    ("Portable power bank 20000mAh PD fast charge", 45.00),
    ("Standing desk mat anti-fatigue", 39.95),
    ("4K webcam with ring light and microphone", 79.99),
    ("Ergonomic office chair mesh back adjustable", 249.00),
    ("Smart thermostat WiFi programmable", 89.00),
    ("Noise cancelling over-ear headphones Bluetooth 5.0", 199.00),
]

BASELINE_PRICE = sum(p for _, p in EVAL_EXAMPLES) / len(EVAL_EXAMPLES)
print(f"Eval set: {len(EVAL_EXAMPLES)} items. Baseline (constant mean): ${BASELINE_PRICE:.2f}")

In [ ]:
def predict_price(summary, model_key):
    if not summary or not summary.strip():
        return 0.0
    client, model_id = get_client(model_key)
    prompt = build_prompt(summary.strip())
    try:
        r = client.chat.completions.create(
            model=model_id,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=15,
        )
        content = (r.choices[0].message.content or "").strip()
        return round(parse_price(content), 2)
    except Exception as e:
        return f"Error: {e}"

def baseline_predict(_summary):
    return round(BASELINE_PRICE, 2)

In [ ]:
def evaluate_mae(model_key, use_baseline=False):
    errors = []
    for summary, actual in EVAL_EXAMPLES:
        pred = baseline_predict(None) if use_baseline else predict_price(summary, model_key)
        if isinstance(pred, (int, float)):
            errors.append(abs(pred - actual))
    if not errors:
        return "Could not compute (predictions failed or errored)."
    mae = sum(errors) / len(errors)
    label = "Baseline (constant mean)" if use_baseline else f"Open-source ({model_key})"
    return f"{label} — MAE over {len(errors)} examples: ${mae:.2f}"

print(evaluate_mae(list(MODELS.keys())[0]))
print(evaluate_mae(None, use_baseline=True))

## Gradio UI

Predict price with an open-source model or compare to baseline; run MAE evaluation.

In [ ]:
def run_eval(model_key, use_baseline):
    return evaluate_mae(model_key, use_baseline=use_baseline)

model_dd = gr.Dropdown(choices=list(MODELS.keys()), value=list(MODELS.keys())[0], label="Model")

with gr.Blocks(title="Week 7 — Open-Source Pricer") as demo:
    gr.Markdown("## Week 7 — Base open-source model price prediction")
    with gr.Row():
        inp = gr.Textbox(label="Product summary", lines=3, placeholder="e.g. Wireless Bluetooth earbuds, 24hr battery...")
        out = gr.Number(label="Predicted price ($)")
    with gr.Row():
        model_dd.render()
        btn = gr.Button("Predict")
    btn.click(fn=lambda s, m: predict_price(s, m), inputs=[inp, model_dd], outputs=out)
    gr.Markdown("### Evaluation (Week 7 style)")
    with gr.Row():
        eval_btn = gr.Button("MAE: Open-source model")
        baseline_btn = gr.Button("MAE: Baseline (constant mean)")
    eval_out = gr.Textbox(label="Result")
    eval_btn.click(fn=lambda m: run_eval(m, False), inputs=[model_dd], outputs=eval_out)
    baseline_btn.click(fn=lambda: run_eval(None, True), inputs=[], outputs=eval_out)

demo.launch()